# Composing your own module

## Setting up DSPy

### Load environment variables

In [1]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [2]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [3]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-4.1-nano",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## `CodeAct` Module Composition

It is composed in the following way:

- The `ReAct` module uses `ChainOfThought` as it's final step.
- `CodeAct` extends both `ReAct` and `ProgramOfThought`.

.. as shown in the figure below:

![](../assets/CodeAct.png)

## Composing modules to build an ensemble

In the same way `CodeAct` Module is composed, we will composed our own `HaikuEnsemble`.

LLM-written poems are a roll of the dice. Sometimes their haikus are evocative; other times they’re predictable and bland. To increase our program’s odds of success, we’re going to roll the dice several times, then select the best candidate.

Here’s what this looks like:

In [4]:
class HaikuEnsemble(dspy.Module):
    def __init__(self, n: int = 3):
        super().__init__()
        self.n = n  
        # Module 1 generates several haikus
        self.writer = dspy.ChainOfThought(
            "location, season, mood, num_haikus: int -> haikus: list[str]",
        )
        # Module 2 picks the most evocative
        self.judge = dspy.ChainOfThought(
            "location, season, mood, candidates: list[str] -> most_evocative_index: int"
        )

    def forward(self, location: str, season: str, mood: str) -> dspy.Prediction:
        candidates = self.writer(
            location=location, season=season, mood=mood, num_haikus=self.n,
        ).haikus
        verdict = self.judge( 
            location=location, season=season, mood=mood, candidates=candidates,
        )
        return dspy.Prediction(
            haiku=candidates[verdict.most_evocative_index],
            candidates=candidates,
            reasoning=verdict.reasoning,
        )

When constructing a module, we need to write two functions:

1. `__init__` sets up our initial state and defines our submodules.
2. `forward` handles what happens when we call our program, accepting inputs and shepherding through our submodules before returning an assembled output.

Our `HaikuEnsemble` defines two submodules in `__init__`.

1. `writer` is similar to our last `ReAct` program. We’ve added a new input field, `num_haikus`, specifying how many haikus we want the model to draft. And we’ve changed our output field to return a `list` of strings.
2. `judge` is entirely new. It accepts the `location`, `season`, and `mood` inputs in addition to the candidate `haikus`. It selects the most evocative of the bunch.

When we call this program, our `forward` method runs each module in sequence, then returns a single `dspy.Prediction` object containing our results.

Let's call this module to see which haiku gets selected:

In [5]:
ensemble = HaikuEnsemble(n=3)
result = ensemble(location="Bodega Bay", season="autumn", mood="inspired")

Let's see the candidates:

In [11]:
for c in result.candidates:
    print(c)
    print("-"*10)

Crimson leaves fall soft,
Waves whisper tales of the sea,
Autumn's quiet dance.
----------
Seagulls soar above,
Golden horizon melting,
Inspired by the breeze.
----------
Seaside's gentle hum,
Colors fade into the waves,
Creative spirits.
----------


The model's reasoning:

In [7]:
print(result.reasoning)

The candidates are poetic lines that evoke imagery related to Bodega Bay during autumn, with themes aligned to the mood of being inspired. The second candidate explicitly mentions seagulls, the sea, and a golden horizon, making a direct connection to the seaside and the inspiring nature of the location. The first focuses on autumn leaves and waves, while the third emphasizes the colors fading and creative spirits, which are also evocative. However, the second candidate most vividly captures the seaside environment combined with the season and inspiration, making it the most evocative choice.


And finally, the haiku selected:

In [8]:
print(result.haiku)

Seagulls soar above,
Golden horizon melting,
Inspired by the breeze.
